## Loading in the Data

### Link to original data if you forget: https://archive.ics.uci.edu/dataset/73/mushroom

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
# List of feature names can be found in agaricus-lepiota.names
feature_names = ["cap-shape", "cap-surface", "cap-color", "bruises", "odor", "gill-attachment",
                 "gill-spacing", "gill-size", "gill-color", "stalk-shape", "stalk-root", "stalk-surface-above-ring",
                 "stalk-surface-below-ring", "stalk-color-above-ring", "stalk-color-below-ring", "veil-type", "veil-color",
                 "ring-number", "ring-type", "spore-print-color", "population", "habitat"]
data = pd.read_csv("mushroom/agaricus-lepiota.data", delimiter = ",", names = feature_names)
target = pd.Series(data.index) # Target vector with classes e, p (edible and poisonous)

In [3]:
# The stalk-root feature only has data for about 70% of the entries
# so it may not be a good predictive feature

len(data[data['stalk-root'] == '?'])/len(data)
# This is actually stated in the names file but I didn't notice until later.

# Cleaning the data so it does not include values with no stalk-root info
data = data.reset_index()
data = data.drop("index", axis = 1)

# Removing bad indexes here
drps = data[data['stalk-root'] == '?'].index
clean_df = data.drop(drps) # Cleaned original data
clean_target = target.drop(drps) # Matching target data
num_tar = clean_target.map({'p': 1, 'e': 0})

In [44]:
hot_df = pd.get_dummies(clean_df, drop_first = True).astype(int) # one-hot encoding the data

#Splitting the data for training and test and then training the model
X_train, X_test, y_train, y_test = train_test_split(hot_df, num_tar, test_size = 0.3, random_state = 50)
log_reg = LogisticRegression(random_state = 50)
model = log_reg.fit(X_train, y_train)

# Testing the model
y_pred = model.predict(X_test)
print(f" The test accuracy of this model is {np.mean(y_pred == y_test)*100}%!") 

 The test accuracy of this model is 100.0%!


In [45]:
# Examining which features are most important, which can be dropped
coefs = model.coef_[0] # Grab coefficients from model and evaluate importance based on abs value
threshhold = 2.5 # Arbitrarily chosen threshold for determining feature importance
rel_features = np.where(np.abs(model.coef_[0]) > threshhold)[0] 
irrel_features = np.where(np.abs(model.coef_[0]) < threshhold)[0]
rel_df = hot_df.drop(hot_df.columns[irrel_features], axis = 1)

# Using sklearn train_test_split and log regression
X_train, X_test, y_train, y_test = train_test_split(rel_df, num_tar, test_size = 0.3, random_state = 50)
log_reg = LogisticRegression(random_state = 50)
spar_model = log_reg.fit(X_train, y_train)

# Testing the model
y_pred = spar_model.predict(X_test)
print(f" The test accuracy of this model is {np.mean(y_pred == y_test)*100}%!")

 The test accuracy of this model is 99.11452184179457%!


In [46]:
print(spar_model.coef_)
print(rel_features)
rel_df

[[ 5.72357542  7.56402611 -1.9000313   5.98877477  6.36885734]]
[16 17 20 21 63]


,odor_c,odor_f,odor_n,odor_p,spore-print-color_r
0,0,0,0,1,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,1,0
4,0,0,1,0,0
...,...,...,...,...,...
7986,0,0,1,0,0
8001,0,0,1,0,0
8038,0,0,1,0,0
8095,0,0,0,0,0
